# 01 Forward Process

Goal: understand the noise schedule and implement the forward noising process.

In this notebook we first visualize the schedule terms from `src/schedule.py`. Later we will use them to implement `q_sample`, which turns a clean image `x_0` into a noisy image `x_t`.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

sys.path.insert(0, str(PROJECT_DIR))
PROJECT_DIR


In [ ]:
import matplotlib.pyplot as plt
import torch

from src.schedule import make_beta_schedule

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))


## 1. Build a Linear Beta Schedule

`beta_t` controls how much noise is added at each step. DDPM commonly starts with a tiny beta and slowly increases it.

From beta we compute:

- `alpha_t = 1 - beta_t`: how much signal remains at step `t`.
- `alpha_bar_t = prod(alpha_0 ... alpha_t)`: cumulative signal retention.
- `sqrt(alpha_bar_t)`: coefficient of the clean image `x_0`.
- `sqrt(1 - alpha_bar_t)`: coefficient of the noise.

In [ ]:
T = 300
schedule = make_beta_schedule(T)

for name, values in schedule.items():
    print(f"{name:28s}", values.shape, values[:3].tolist(), "...", values[-3:].tolist())


## 2. Visualize the Schedule

The most important intuition is: as `t` grows, the clean-image coefficient goes down and the noise coefficient goes up.

In [ ]:
steps = torch.arange(T)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
axes = axes.flatten()

plots = [
    ("betas", "beta_t"),
    ("alpha_bars", "alpha_bar_t"),
    ("sqrt_alpha_bars", "sqrt(alpha_bar_t), clean image weight"),
    ("sqrt_one_minus_alpha_bars", "sqrt(1 - alpha_bar_t), noise weight"),
]

for ax, (key, title) in zip(axes, plots):
    ax.plot(steps, schedule[key].cpu())
    ax.set_title(title)
    ax.set_xlabel("timestep t")
    ax.grid(True, alpha=0.3)

plt.tight_layout();


## 3. Inspect a Few Timesteps

This table shows how much clean signal and noise are mixed at selected timesteps.

In [ ]:
selected_steps = torch.tensor([0, 25, 50, 100, 200, T - 1])

print(f"{'t':>5} | {'sqrt_alpha_bar':>16} | {'sqrt_1m_alpha_bar':>18}")
print("-" * 48)
for t in selected_steps:
    clean_weight = schedule["sqrt_alpha_bars"][t].item()
    noise_weight = schedule["sqrt_one_minus_alpha_bars"][t].item()
    print(f"{int(t):5d} | {clean_weight:16.4f} | {noise_weight:18.4f}")


In [ ]:
from torchvision.utils import make_grid

from src.datasets import DatasetConfig, denormalize, make_dataloader
from src.forward_process import q_sample


## 4. Visualize Forward Noising on Images

Now we use `q_sample` to create `x_t` from real images. Each row is one original image; columns show progressively larger timesteps.

This also checks that the same code works for both grayscale MNIST `[B, 1, 28, 28]` and RGB CIFAR-10 `[B, 3, 32, 32]`.

In [ ]:
def visualize_forward_noising(dataset_name, image_size, channels, num_images=3):
    cfg = DatasetConfig(
        name=dataset_name,
        image_size=image_size,
        channels=channels,
        batch_size=max(num_images, 8),
        num_workers=0,
    )
    loader = make_dataloader(cfg, train=True)
    images, labels = next(iter(loader)) # images: [N,C,H,W]

    # x0 has shape [N, C, H, W], even for grayscale MNIST where C=1.
    # x0 的 shape 是 [N, C, H, W]；MNIST 灰度图也保留 channel 维度 C=1。
    x0 = images[:num_images]

    selected_steps = torch.tensor([0, 25, 50, 100, 200, T - 1])

    # We want every image to appear at every selected timestep.
    # Example: num_images=3 and len(selected_steps)=6 gives 18 batch items.
    # repeat_interleave(..., dim=0) repeats each image along the batch dimension:
    # [img0, img1, img2] -> [img0,img0,..., img1,img1,..., img2,img2,...].
    # repeat_interleave()是原地复制某个维度元素，而repeat()则是整个维度复制。注意两个区别
    # 例如，对于一张彩色图片 img=(3,2,2), 如果 repeat_interleave(img,2,dim=0)就会变成了[R,R,G,G,B,B]这种样子
    # 而 img.repeat(2,1,1)则会变成 [R,G,B,R,G,B]这个样子。两个输出的尺寸完全相同[6,2,2]，但是dim=0的复制顺序不同
    # 注意两者都不会增加新维度。
    # 这里就是把 batch 维按照元素复制的方法扩展为 N * num_steps，结果是[img0,img0,img0,img1,img1,img1,img2,img2,img2]
    x0_batch = x0.repeat_interleave(len(selected_steps), dim=0)

    # Match the flattened image batch above:
    # [0,25,50,100,200,299] repeated once per original image.
    # 例如这里结果是[0, 25, 50, 100, 200, 299, 0, 25, 50, 100, 200, 299, 0, 25, 50, 100, 200, 299]
    # 结合上面x0_batch的顺序，我们需要的是“每张图片连续对应多个 timestep”，因此这里采用 repeat()
    t_batch = selected_steps.repeat(num_images)

    print("x0_batch:", tuple(x0_batch.shape), "t_batch:", tuple(t_batch.shape))

    # One random noise tensor per displayed image/timestep pair.
    noise = torch.randn_like(x0_batch)
    noisy_images, _ = q_sample(
        x0=x0_batch,
        t=t_batch,
        sqrt_alpha_bars=schedule["sqrt_alpha_bars"],
        sqrt_one_minus_alpha_bars=schedule["sqrt_one_minus_alpha_bars"],
        noise=noise,
    )

    print(f"{dataset_name}: x0={tuple(x0.shape)}, noisy={tuple(noisy_images.shape)}, labels={labels[:num_images].tolist()}")

    grid = make_grid(denormalize(noisy_images), nrow=len(selected_steps), padding=2)
    plt.figure(figsize=(12, 2.2 * num_images))
    plt.imshow(grid.permute(1, 2, 0))
    plt.axis("off")
    plt.title(f"{dataset_name} forward diffusion: " + "  ".join(f"t={int(t)}" for t in selected_steps))
    plt.show()


### MNIST

Three grayscale examples. The same `q_sample` formula broadcasts over `[B, 1, H, W]`.

In [ ]:
visualize_forward_noising("mnist", image_size=28, channels=1, num_images=3)


### CIFAR-10

Three RGB examples. This verifies the forward process is channel-agnostic and works for `[B, 3, H, W]` as well.

In [ ]:
visualize_forward_noising("cifar10", image_size=32, channels=3, num_images=3)


## 5. What This Means

Forward diffusion does not learn anything. It is a fixed noising process. Training starts in the next step: the model receives `x_t` and `t`, then learns to predict the noise that produced `x_t`.